# IPL Ball-by-Ball Analysis
Dataset: `archive/IPL.csv` — 278,205 deliveries across 18 IPL seasons (2007/08 – 2025)

In [ ]:
import pandas as pd
import numpy as np

## Load Data

In [ ]:
df = pd.read_csv('../archive/IPL.csv', index_col=0, low_memory=False)

# Fix mixed-type columns flagged by pandas
mixed_cols = ['team_reviewed', 'review_decision', 'umpire', 'umpires_call',
              'gender', 'result_type', 'method', 'balls_per_over', 'stage']
df[mixed_cols] = df[mixed_cols].astype(str).replace('nan', pd.NA)

print(f'Loaded {len(df):,} rows × {df.shape[1]} columns')
df.head()

## Schema & Nulls

In [ ]:
print('--- dtypes ---')
print(df.dtypes)
print('\n--- null counts (non-zero only) ---')
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0].sort_values(ascending=False))

## Data Distribution

In [ ]:
# Deliveries per season
print('--- Deliveries per season ---')
print(df['season'].value_counts().sort_index())

In [ ]:
# Numeric summary for key batting/bowling columns
numeric_cols = ['runs_batter', 'runs_extras', 'runs_total', 'over', 'innings']
df[numeric_cols].describe().round(3)

In [ ]:
# Runs distribution
print('--- runs_batter value counts ---')
print(df['runs_batter'].value_counts().sort_index())

print('\n--- Boundary rate (4s + 6s) ---')
boundaries = df['runs_batter'].isin([4, 6]).sum()
valid_balls = df['valid_ball'].sum()
print(f'{boundaries:,} boundaries in {valid_balls:,} valid deliveries ({boundaries/valid_balls:.1%})')

In [ ]:
# Wicket kinds
print('--- Wicket kinds ---')
print(df['wicket_kind'].value_counts())

In [ ]:
# Extra types
print('--- Extra types ---')
print(df['extra_type'].value_counts())

In [ ]:
# Teams
print(f'--- {df["batting_team"].nunique()} unique teams ---')
print(df['batting_team'].value_counts())

In [ ]:
# Top venues
print(f'--- {df["venue"].nunique()} unique venues (top 10) ---')
print(df['venue'].value_counts().head(10))

## Match-Level Aggregation (pre-match features)

Goal: one row per match for **from-scratch** score prediction. Features are all known before a ball is bowled. Target is `inn1_runs`; `inn1_team_won` is a secondary label.

- **Batting-side `players`** = full XI for that team, reconstructed as the union of everyone who batted, bowled, or took a catch/fielded a wicket for that team.
- **Bowling-side `opp_bowlers`** = only players who actually bowled in that innings.
- Rain-affected matches (D/L or no-result) are dropped.
- Team names are mapped to acronyms: current 2026 franchises use their current codes (Delhi Daredevils → DC same as Delhi Capitals; Kings XI Punjab → PBKS same as Punjab Kings). Defunct franchises keep their historical codes (PWI, RPSG, GL, KTK, DCH).

In [ ]:
TEAM_MAP = {
    'Chennai Super Kings': 'CSK',
    'Mumbai Indians': 'MI',
    'Royal Challengers Bangalore': 'RCB',
    'Royal Challengers Bengaluru': 'RCB',
    'Kolkata Knight Riders': 'KKR',
    'Delhi Daredevils': 'DC',
    'Delhi Capitals': 'DC',
    'Kings XI Punjab': 'PBKS',
    'Punjab Kings': 'PBKS',
    'Rajasthan Royals': 'RR',
    'Sunrisers Hyderabad': 'SRH',
    'Gujarat Titans': 'GT',
    'Lucknow Super Giants': 'LSG',
    'Deccan Chargers': 'DCH',
    'Pune Warriors': 'PWI',
    'Rising Pune Supergiant': 'RPSG',
    'Rising Pune Supergiants': 'RPSG',
    'Gujarat Lions': 'GL',
    'Kochi Tuskers Kerala': 'KTK',
}

df_reg = df[df['innings'].isin([1, 2])].copy()

# Drop rain-affected matches: D/L applied or abandoned (no result)
flags = df_reg.groupby('match_id').agg(
    method=('method', lambda s: s.dropna().iloc[0] if s.dropna().size else None),
    result_type=('result_type', lambda s: s.dropna().iloc[0] if s.dropna().size else None),
)
drop_ids = set(flags[(flags['method'] == 'D/L') | (flags['result_type'] == 'no result')].index)
df_reg = df_reg[~df_reg['match_id'].isin(drop_ids)]
print(f'Dropped {len(drop_ids)} rain-affected matches')

def split_fielders(s):
    if pd.isna(s):
        return []
    return [x.strip() for x in str(s).split(',') if x.strip()]

rows = []
for mid, g in df_reg.groupby('match_id', sort=False):
    inn1 = g[g['innings'] == 1]
    inn2 = g[g['innings'] == 2]
    if inn1.empty or inn2.empty:
        continue

    t1 = inn1['batting_team'].iloc[0]
    t2 = inn2['batting_team'].iloc[0]
    winner = g['match_won_by'].iloc[0]

    t1_squad = sorted(
        set(inn1['batter'].dropna()) | set(inn1['non_striker'].dropna())
        | set(inn2['bowler'].dropna())
        | {f for lst in inn2['fielders'].map(split_fielders) for f in lst}
    )
    t2_squad = sorted(
        set(inn2['batter'].dropna()) | set(inn2['non_striker'].dropna())
        | set(inn1['bowler'].dropna())
        | {f for lst in inn1['fielders'].map(split_fielders) for f in lst}
    )

    rows.append({
        'match_id':          mid,
        'season':            g['season'].iloc[0],
        'venue':             g['venue'].iloc[0],
        'city':              g['city'].iloc[0],
        'toss_winner':       TEAM_MAP.get(g['toss_winner'].iloc[0], g['toss_winner'].iloc[0]),
        'toss_decision':     g['toss_decision'].iloc[0],
        'inn1_batting_team': TEAM_MAP.get(t1, t1),
        'inn1_players':      '|'.join(t1_squad),
        'inn1_opp_bowlers':  '|'.join(sorted(set(inn1['bowler'].dropna()))),
        'inn1_runs':         int(inn1['runs_total'].sum()),
        'inn2_batting_team': TEAM_MAP.get(t2, t2),
        'inn2_players':      '|'.join(t2_squad),
        'inn2_opp_bowlers':  '|'.join(sorted(set(inn2['bowler'].dropna()))),
        'inn1_team_won':     int(winner == t1),
    })

matches = pd.DataFrame(rows)
print(f"{len(matches):,} completed matches")
print(f"inn1_team_won rate: {matches['inn1_team_won'].mean():.3f}")
matches.head()

In [ ]:
# Sanity: squad sizes should mostly be 11 (12 allowed since 2023 impact-player rule)
sizes = pd.DataFrame({
    'inn1_squad_size': matches['inn1_players'].str.split('|').str.len(),
    'inn2_squad_size': matches['inn2_players'].str.split('|').str.len(),
    'inn1_bowlers':    matches['inn1_opp_bowlers'].str.split('|').str.len(),
    'inn2_bowlers':    matches['inn2_opp_bowlers'].str.split('|').str.len(),
})
print(sizes.describe().round(2))

In [ ]:
out_path = '../archive/matches_aggregated.csv'
matches.to_csv(out_path, index=False)
print(f'Wrote {out_path}')

## Per-Player Career Stats

Career batting and bowling stats for every player in the ball-by-ball data. Fed as extra per-player features into the Stage 1 embedding model (Option B). Scope matches `matches_aggregated.csv` — innings 1 and 2, rain-affected matches excluded.

Columns (redundant counts dropped — strike_rate subsumes balls_faced, economy subsumes runs_conceded):
- `runs`, `strike_rate` — batting volume + rate
- `wickets`, `balls_bowled`, `economy` — bowling volume + workload + rate

Players who only batted get 0s in the bowling columns, and vice versa.

**Leakage note:** these are career totals through 2025. Using them directly in Stage 1 for a 2012 match uses future information. Fine as a starting point; for a clean pipeline regenerate as-of each match date.

In [ ]:
# Batting stats: runs (volume) + strike rate (rate)
bat_stats = (df_reg.groupby('batter', sort=False)
             .agg(runs=('runs_batter', 'sum'),
                  _balls=('balls_faced', 'sum'))
             .reset_index()
             .rename(columns={'batter': 'player'}))
bat_stats['strike_rate'] = (bat_stats['runs'] * 100.0 / bat_stats['_balls']).where(
    bat_stats['_balls'] > 0, 0.0)
bat_stats = bat_stats.drop(columns='_balls')

# Bowling stats: wickets (volume) + balls bowled (workload) + economy (rate)
bowl_stats = (df_reg.groupby('bowler', sort=False)
              .agg(wickets=('bowler_wicket', 'sum'),
                   _runs=('runs_bowler', 'sum'),
                   balls_bowled=('valid_ball', 'sum'))
              .reset_index()
              .rename(columns={'bowler': 'player'}))
bowl_stats['economy'] = (bowl_stats['_runs'] * 6.0 / bowl_stats['balls_bowled']).where(
    bowl_stats['balls_bowled'] > 0, 0.0)
bowl_stats = bowl_stats.drop(columns='_runs')

player_stats = (bat_stats.merge(bowl_stats, on='player', how='outer')
                .fillna(0)
                .sort_values('player')
                .reset_index(drop=True))

for c in ['runs', 'wickets', 'balls_bowled']:
    player_stats[c] = player_stats[c].astype(int)

player_stats = player_stats[['player',
                             'runs', 'strike_rate',
                             'wickets', 'balls_bowled', 'economy']]

print(f'{len(player_stats):,} players')
print('\nTop 5 by runs:')
print(player_stats.sort_values('runs', ascending=False).head(5).to_string(index=False))
print('\nTop 5 by wickets:')
print(player_stats.sort_values('wickets', ascending=False).head(5).to_string(index=False))

In [ ]:
stats_path = '../archive/player_stats.csv'
player_stats.to_csv(stats_path, index=False)
print(f'Wrote {stats_path}')